# scikit-learn in one pattern

The [first lecture](01-lecture-lifecycle-and-linear-regression.ipynb) fitted a linear regression with scikit-learn. This short lecture steps back and shows the syntax of the library itself. Every model in scikit-learn, from linear regression to random forests, follows the same pattern: create the object with its settings, `fit` it to training data, then `predict` or `score` on new data. Preprocessing steps follow a sibling pattern (`fit`, then `transform`), and both combine into pipelines that again behave like a single model. Once you know the pattern, every later session of the course (classification, trees, clustering, text models) only adds new class names. The examples use the short-stay Berlin listings of Inside Airbnb (snapshot of 26 June 2026, CC BY 4.0), as in the first lecture.

```mermaid
flowchart LR
    TR["training data<br/>X_train, y_train"] --> F["fit(X_train, y_train)<br/>learns attributes_"]
    F --> T["transformer:<br/>transform(X)"]
    F --> P["predictor:<br/>predict(X_new),<br/>score(X_test, y_test)"]
    TE["new or test data<br/>X_test"] --> T
    TE --> P
    T --> P
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## The one pattern every estimator follows

### Concept

scikit-learn calls every object that learns from data an **estimator**. Using one always takes the same steps:

1. **Data.** The features are a 2-D table `X` (one row per observation, one column per feature), the target is a 1-D array `y` with one value per row. A pandas DataFrame and a Series work directly.
2. **Construct.** `model = LinearRegression()` creates the object. Settings chosen by you, the **hyperparameters**, are arguments of the constructor, for example `KNeighborsRegressor(n_neighbors=10)`. Nothing is learned yet.
3. **Fit.** `model.fit(X, y)` learns from the data and returns the model itself.
4. **Use.** `model.predict(X_new)` returns predictions; `model.score(X_test, y_test)` returns a default metric (R² for regressors, accuracy for classifiers).

Everything the model has **learned** is stored in attributes whose names end in an underscore: `coef_`, `intercept_`, `classes_`, `feature_names_in_`, `n_features_in_`. The trailing underscore tells you that the attribute exists only after `fit`. The hyperparameters are read with `get_params()` and changed with `set_params()`.

### Why it matters

Because all models share these method names, you can swap one model for another without rewriting the code around it, and tools such as cross-validation and grid search (Session 7) work with any model. The design is described in Buitinck et al. (2013).

### How it works in Python

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
short = listings[listings["price"].notna() & listings["minimum_nights"].lt(28)].copy()   # comparable prices
short["log_price"] = np.log(short["price"])
short["km_to_centre"] = 111.2 * np.hypot(short["latitude"] - 52.5219,                       # Alexanderplatz
                                         (short["longitude"] - 13.4132) * np.cos(np.radians(52.52)))

X = short[["accommodates", "km_to_centre"]]          # 2-D: rows = listings, columns = features
y = short["log_price"]                               # 1-D: one target value per row
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)

model = LinearRegression()                           # 1. construct (no learning yet)
model.fit(X_train, y_train)                          # 2. learn from the training data
print(model.predict(X_test.head(3)))                 # 3. predict for new rows
print(round(model.score(X_test, y_test), 3))         # 4. R² on the test data

(5360, 2) (5360,)
[5.18743681 5.20505098 5.0550755 ]
0.35


In [3]:
# learned attributes end in "_"; hyperparameters come from get_params()
print(model.feature_names_in_, model.coef_.round(3), round(model.intercept_, 3))
print(model.get_params())

['accommodates' 'km_to_centre'] [ 0.153 -0.021] 4.625
{'copy_X': True, 'fit_intercept': True, 'n_jobs': None, 'positive': False, 'tol': 1e-06}


`fit` returns the model, so `LinearRegression().fit(X_train, y_train).predict(X_test)` also works. Calling `predict` before `fit` raises a `NotFittedError`.

> **Note.** `X` must be 2-D even with a single feature: `short[["accommodates"]]` (a DataFrame), not `short["accommodates"]` (a Series). With a NumPy array, use `x.reshape(-1, 1)`.

## The same code with other models

### Concept

The four steps do not depend on the model. The loop below fits four very different regressors with identical code: a linear model, *k*-nearest neighbours (Session 8), a decision tree and a random forest (Session 10). Only the constructor changes. Classifiers follow the same pattern; they add `predict_proba`, which returns one probability per class, and `classes_`, the order of those classes.

### How it works in Python

In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

models = [LinearRegression(),
          KNeighborsRegressor(n_neighbors=20),
          DecisionTreeRegressor(max_depth=5, random_state=0),
          RandomForestRegressor(n_estimators=100, min_samples_leaf=5, random_state=0)]
for model in models:                                  # the same five lines for every model
    model.fit(X_train, y_train)
    print(f"{type(model).__name__:22s} test R² {model.score(X_test, y_test):.3f}")

LinearRegression       test R² 0.350
KNeighborsRegressor    test R² 0.390
DecisionTreeRegressor  test R² 0.385


RandomForestRegressor  test R² 0.375


In [5]:
from sklearn.linear_model import LogisticRegression

# a classifier: is a short-stay listing an entire home? (features: guests, distance, log price)
Xc = short[["accommodates", "km_to_centre", "log_price"]]
yc = short["room_type"].eq("Entire home/apt")
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)

clf = LogisticRegression(max_iter=1000).fit(Xc_train, yc_train)
print(clf.classes_)                                   # order of the columns of predict_proba
print(clf.predict(Xc_test.head(3)))                   # predicted classes
print(clf.predict_proba(Xc_test.head(3)).round(3))    # one probability per class
print(round(clf.score(Xc_test, yc_test), 3))          # accuracy

[False  True]
[ True  True  True]
[[0.026 0.974]
 [0.282 0.718]
 [0.347 0.653]]
0.82


### In practice

- A first comparison of several models with the same split, as above, is a common start of a project; Session 7 replaces the single split by cross-validation.
- `set_params` changes a hyperparameter of an existing object, for example `models[1].set_params(n_neighbors=50)`; the model must then be fitted again.

## Transformers: fit, transform, fit_transform

### Concept

A **transformer** learns something from data and then changes data with it. `StandardScaler` learns the mean and standard deviation of each column (`mean_`, `scale_`) and transforms each value to (x − mean) / sd. `OneHotEncoder` learns the categories of a column (`categories_`) and turns each category into a 0/1 column. The methods are `fit(X)`, `transform(X)` and the shortcut `fit_transform(X)`.

**Fit on the training data only**, then transform training and test data with what was learned. A scaler fitted on all rows uses information from the test rows; the test score is then no longer a fair estimate (data leakage, Session 7). `set_output(transform="pandas")` makes transformers return DataFrames with column names instead of NumPy arrays.

### How it works in Python

In [6]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

scaler = StandardScaler().set_output(transform="pandas")
scaler.fit(X_train)                                   # learn mean and sd from the training rows only
print(scaler.mean_.round(2), scaler.scale_.round(2))
print(scaler.transform(X_test).head(3).round(2))      # apply them to the test rows

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False).set_output(transform="pandas")
rooms = encoder.fit_transform(short[["room_type"]])   # fit and transform in one step
print(encoder.categories_)
rooms.head(3)

[3.58 5.23] [2.31 3.99]
       accommodates  km_to_centre
8989           0.18         -0.69
5392           0.18         -0.90
11144         -0.25         -0.94
[array(['Entire home/apt', 'Hotel room', 'Private room', 'Shared room'],
      dtype=object)]


,room_type_Entire home/apt,room_type_Hotel room,room_type_Private room,room_type_Shared room
0,1.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0


## Composition: pipelines and column transformers

### Concept

A **pipeline** chains transformers and a final model: `make_pipeline(StandardScaler(), KNeighborsRegressor())`. Calling `fit` fits each step on the output of the previous one; `predict` transforms new data with the fitted steps and then predicts. A pipeline is itself an estimator, so it has the same `fit`, `predict` and `score` and can be used wherever a model can. It also makes the "fit only on training data" rule automatic.

A **ColumnTransformer** applies different transformers to different columns, for example scaling to numeric columns and one-hot encoding to categorical ones, and puts the results side by side.

### How it works in Python

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

features = ["accommodates", "km_to_centre", "room_type", "district"]
X = short[features]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocess = ColumnTransformer([
    ("numeric", StandardScaler(), ["accommodates", "km_to_centre"]),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), ["room_type", "district"]),
])
for final in [LinearRegression(), KNeighborsRegressor(n_neighbors=20)]:
    pipe = make_pipeline(preprocess, final)          # a pipeline is an estimator: same fit / score
    pipe.fit(X_train, y_train)
    print(f"{type(final).__name__:20s} test R² {pipe.score(X_test, y_test):.3f}")
print(pipe.named_steps)                              # the fitted steps, by name

LinearRegression     test R² 0.501


KNeighborsRegressor  test R² 0.487
{'columntransformer': ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                 ['accommodates', 'km_to_centre']),
                                ('categorical',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['room_type', 'district'])]), 'kneighborsregressor': KNeighborsRegressor(n_neighbors=20)}


## Model selection uses the same objects

### Concept

The tools for splitting and evaluating take an estimator, or a whole pipeline, as their first argument and call its `fit` and `score` themselves:

- `train_test_split(X, y, ...)` splits the rows once (used above);
- `cross_val_score(model, X, y, cv=5)` fits and scores the model on five different splits (Session 7);
- `GridSearchCV(model, param_grid, cv=5)` tries several hyperparameter values with cross-validation and is then itself an estimator with `fit`, `predict` and `best_params_` (Session 7). In a pipeline, a parameter is addressed as `<step name>__<parameter>`.

Classification metrics beyond accuracy follow in Session 8.

### How it works in Python

In [8]:
from sklearn.model_selection import GridSearchCV, cross_val_score

pipe = make_pipeline(preprocess, KNeighborsRegressor())
scores = cross_val_score(pipe, X_train, y_train, cv=5)          # R² on five folds of the training data
print(scores.round(3), round(scores.mean(), 3))

search = GridSearchCV(pipe, {"kneighborsregressor__n_neighbors": [5, 20, 50]}, cv=5)
search.fit(X_train, y_train)                                      # same fit as any model
print(search.best_params_, round(search.score(X_test, y_test), 3))

[0.471 0.481 0.478 0.444 0.471] 0.469


{'kneighborsregressor__n_neighbors': 20} 0.487


## Cheat sheet

| Task | Code | Notes |
|---|---|---|
| Features and target | `X = df[["a", "b"]]`, `y = df["t"]` | `X` 2-D, `y` 1-D |
| Split | `train_test_split(X, y, test_size=0.2, random_state=42)` | `stratify=y` for classes |
| Create a model | `model = KNeighborsRegressor(n_neighbors=20)` | hyperparameters in the constructor |
| Learn | `model.fit(X_train, y_train)` | returns the model |
| Predict | `model.predict(X_new)`, `model.predict_proba(X_new)` | `predict_proba` for classifiers |
| Evaluate | `model.score(X_test, y_test)` | R² (regression), accuracy (classification) |
| Learned values | `coef_`, `intercept_`, `classes_`, `feature_names_in_` | end in `_`, exist after `fit` |
| Settings | `get_params()`, `set_params(n_neighbors=50)` | refit after changing |
| Transform | `fit(X_train)`, `transform(X)`, `fit_transform(X_train)` | fit on training data only |
| DataFrames out | `.set_output(transform="pandas")` | column names kept |
| Chain steps | `make_pipeline(StandardScaler(), model)` | a pipeline is an estimator |
| Columns by type | `ColumnTransformer([("num", StandardScaler(), num_cols), ("cat", OneHotEncoder(), cat_cols)])` | |
| Cross-validate | `cross_val_score(model, X, y, cv=5)` | Session 7 |
| Tune | `GridSearchCV(model, {"step__param": [...]}, cv=5)` | Session 7 |

## Check your understanding

1. Why does `model.fit(short["accommodates"], y)` fail, and how do you fix it?
2. What is the difference between `n_neighbors` and `coef_`? How can you tell from the name alone?
3. Why should a `StandardScaler` be fitted on the training rows only, and how does a pipeline take care of that?
4. You replace `LinearRegression()` by `RandomForestRegressor()` in the loop above. Which lines of code change?
5. A classifier returns `predict_proba` with two columns. How do you find out which column belongs to which class?

## Further reading

- scikit-learn developers (2026). *Getting started*. <https://scikit-learn.org/stable/getting_started.html>
- scikit-learn developers (2026). *Glossary of common terms and API elements*. <https://scikit-learn.org/stable/glossary.html>
- Buitinck, L., Louppe, G., Blondel, M., et al. (2013). API design for machine learning software: experiences from the scikit-learn project. *ECML PKDD Workshop: Languages for Data Mining and Machine Learning*, 108–122. <https://arxiv.org/abs/1309.0238>